In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("nqa112/vizwiz-2023-edition")

print("Path to dataset files:", path)

100%|██████████| 17.5G/17.5G [03:01<00:00, 103MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1


In [4]:
!ls /root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json

/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json


In [6]:
import json
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Đọc dữ liệu
with open('/content/default_pred.json', 'r') as f:
    predictions = json.load(f)

with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json', 'r') as f:
    ground_truths = json.load(f)

# Tạo dictionary cho ground truth để dễ tra cứu
gt_dict = {item['image']: item['answers'] for item in ground_truths}

# Khởi tạo
scores = []
smoothie = SmoothingFunction().method4

for pred in predictions:
    image_id = pred['image']
    pred_answer = pred['answer']

    # Tokenize
    pred_tokens = pred_answer.lower().split()

    # Lấy các ground truth answers
    references = [ans['answer'].lower().split() for ans in gt_dict.get(image_id, []) if 'answer' in ans]

    # Tính BLEU cho từng câu
    if references:
        bleu = sentence_bleu(references, pred_tokens, smoothing_function=smoothie)
        scores.append(bleu)

# Tính trung bình BLEU score
average_bleu = sum(scores) / len(scores) if scores else 0.0
print(f"Average BLEU score: {average_bleu:.4f}")


Average BLEU score: 0.6981


In [7]:
import json
from collections import Counter

# Đọc dữ liệu
with open('/content/default_pred.json') as f:
    predictions = json.load(f)

with open('/root/.cache/kagglehub/datasets/nqa112/vizwiz-2023-edition/versions/1/Annotations/val.json') as f:
    ground_truths = json.load(f)

# Tạo dict tra cứu ground truth theo image_id
gt_dict = {item['image']: item['answers'] for item in ground_truths}

accuracies = []

for pred in predictions:
    image_id = pred['image']
    pred_answer = pred['answer'].strip().lower()

    # Lấy các câu trả lời ground truth (chuyển về chữ thường và strip)
    gt_answers = [
        ans['answer'].strip().lower()
        for ans in gt_dict.get(image_id, [])
        if 'answer' in ans
    ]

    # Đếm số lần dự đoán trùng với các câu trả lời
    matching_count = sum(1 for ans in gt_answers if ans == pred_answer)

    # Tính accuracy chuẩn VizWiz
    acc = min(matching_count / 3, 1.0)
    accuracies.append(acc)

# Tính trung bình
average_accuracy = sum(accuracies) / len(accuracies) if accuracies else 0.0
print(f"VizWiz-style VQA Accuracy: {average_accuracy:.4f}")


VizWiz-style VQA Accuracy: 0.6132
